In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("../Airlinedataset.csv")

# Features used by the validated model
features = [
    "days_to_departure",
    "seats_remaining",
    "historical_demand",
    "competitor_price",
    "booking_velocity",
    "is_weekend",
    "flight_capacity"
]

target = "ticket_price"

X = df[features]
y = df[target]

# Same split used throughout the project
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Train validated baseline model
model = LinearRegression()
model.fit(X_train, y_train)

print("Pricing model trained successfully.")

Pricing model trained successfully.


In [2]:
# Generate model predictions
test_predictions = model.predict(X_test)

# Add predictions to a copy of the test data
pricing_test = X_test.copy()
pricing_test["predicted_price"] = test_predictions

pricing_test.head()

,days_to_departure,seats_remaining,historical_demand,competitor_price,booking_velocity,is_weekend,flight_capacity,predicted_price
521,51,170,49,3882,4,0,180,5892.058594
737,25,92,44,4167,29,1,180,9464.175763
740,59,65,41,7732,19,1,180,9271.857886
660,33,90,79,8537,28,0,220,10801.921395
411,46,175,26,4736,13,1,150,6379.478406


In [3]:
# Check the predicted price range
print("Minimum predicted price:", round(pricing_test["predicted_price"].min(), 2))
print("Maximum predicted price:", round(pricing_test["predicted_price"].max(), 2))
print("Average predicted price:", round(pricing_test["predicted_price"].mean(), 2))

Minimum predicted price: 5777.19
Maximum predicted price: 12047.4
Average predicted price: 9302.41


In [4]:
def recommend_price(
    days_to_departure,
    seats_remaining,
    historical_demand,
    competitor_price,
    booking_velocity,
    is_weekend,
    flight_capacity
):
    flight_state = pd.DataFrame([{
        "days_to_departure": days_to_departure,
        "seats_remaining": seats_remaining,
        "historical_demand": historical_demand,
        "competitor_price": competitor_price,
        "booking_velocity": booking_velocity,
        "is_weekend": is_weekend,
        "flight_capacity": flight_capacity
    }])

    predicted_price = model.predict(flight_state)[0]

    return round(predicted_price, 2)

In [5]:
normal_price = recommend_price(
    days_to_departure=30,
    seats_remaining=90,
    historical_demand=60,
    competitor_price=6250,
    booking_velocity=16,
    is_weekend=0,
    flight_capacity=180
)

print("Recommended price:", normal_price)

Recommended price: 9214.99


In [6]:
high_demand_price = recommend_price(
    days_to_departure=5,
    seats_remaining=20,
    historical_demand=90,
    competitor_price=7000,
    booking_velocity=28,
    is_weekend=1,
    flight_capacity=180
)

print("High-demand scenario price:", high_demand_price)

High-demand scenario price: 12905.74


In [7]:
low_demand_price = recommend_price(
    days_to_departure=45,
    seats_remaining=150,
    historical_demand=30,
    competitor_price=5500,
    booking_velocity=5,
    is_weekend=0,
    flight_capacity=180
)

print("Low-demand scenario price:", low_demand_price)

Low-demand scenario price: 6574.36


In [10]:
scenarios = pd.DataFrame([
    {
        "scenario": "Low demand",
        "days_to_departure": 45,
        "seats_remaining": 150,
        "historical_demand": 30,
        "competitor_price": 5500,
        "booking_velocity": 5,
        "is_weekend": 0,
        "flight_capacity": 180
    },
    {
        "scenario": "Normal demand",
        "days_to_departure": 30,
        "seats_remaining": 90,
        "historical_demand": 60,
        "competitor_price": 6250,
        "booking_velocity": 16,
        "is_weekend": 0,
        "flight_capacity": 180
    },
    {
        "scenario": "High demand",
        "days_to_departure": 5,
        "seats_remaining": 20,
        "historical_demand": 90,
        "competitor_price": 7000,
        "booking_velocity": 28,
        "is_weekend": 1,
        "flight_capacity": 180
    }
])

scenarios["recommended_price"] = model.predict(
    scenarios[features]
)

scenarios[[
    "scenario",
    "days_to_departure",
    "seats_remaining",
    "historical_demand",
    "booking_velocity",
    "recommended_price"
]]

,scenario,days_to_departure,seats_remaining,historical_demand,booking_velocity,recommended_price
0,Low demand,45,150,30,5,6574.361330
1,Normal demand,30,90,60,16,9214.989307
2,High demand,5,20,90,28,12905.738746


In [11]:
scenarios["recommended_price"] = scenarios["recommended_price"].round(2)

print(
    scenarios[
        ["scenario", "recommended_price"]
    ].to_string(index=False)
)

     scenario  recommended_price
   Low demand            6574.36
Normal demand            9214.99
  High demand           12905.74


In [12]:
def dynamic_price_recommendation(flight_state):
    """
    Generate a model-based dynamic ticket price recommendation.

    Parameters
    ----------
    flight_state : dict
        Current flight-state variables.

    Returns
    -------
    float
        Model-predicted ticket price.
    """

    required_features = [
        "days_to_departure",
        "seats_remaining",
        "historical_demand",
        "competitor_price",
        "booking_velocity",
        "is_weekend",
        "flight_capacity"
    ]

    input_data = pd.DataFrame([flight_state])[required_features]

    predicted_price = model.predict(input_data)[0]

    return round(float(predicted_price), 2)

In [13]:
flight = {
    "days_to_departure": 10,
    "seats_remaining": 35,
    "historical_demand": 85,
    "competitor_price": 6800,
    "booking_velocity": 25,
    "is_weekend": 1,
    "flight_capacity": 180
}

recommended_price = dynamic_price_recommendation(flight)

print("Dynamic Pricing Recommendation")
print("--------------------------------")
print(f"Recommended ticket price: ₹{recommended_price:,.2f}")

Dynamic Pricing Recommendation
--------------------------------
Recommended ticket price: ₹12,233.23


In [14]:
base_flight = {
    "days_to_departure": 30,
    "seats_remaining": 90,
    "historical_demand": 60,
    "competitor_price": 6250,
    "booking_velocity": 16,
    "is_weekend": 0,
    "flight_capacity": 180
}

# Baseline
base_price = dynamic_price_recommendation(base_flight)

# Change only days to departure
earlier_departure = base_flight.copy()
earlier_departure["days_to_departure"] = 10

# Change only seats remaining
fewer_seats = base_flight.copy()
fewer_seats["seats_remaining"] = 30

# Change only historical demand
higher_demand = base_flight.copy()
higher_demand["historical_demand"] = 90

# Change only competitor price
higher_competitor = base_flight.copy()
higher_competitor["competitor_price"] = 8000

# Change only booking velocity
higher_velocity = base_flight.copy()
higher_velocity["booking_velocity"] = 28

sensitivity_results = pd.DataFrame({
    "scenario": [
        "Baseline",
        "Earlier departure",
        "Fewer seats",
        "Higher historical demand",
        "Higher competitor price",
        "Higher booking velocity"
    ],
    "recommended_price": [
        base_price,
        dynamic_price_recommendation(earlier_departure),
        dynamic_price_recommendation(fewer_seats),
        dynamic_price_recommendation(higher_demand),
        dynamic_price_recommendation(higher_competitor),
        dynamic_price_recommendation(higher_velocity)
    ]
})

sensitivity_results["price_change"] = (
    sensitivity_results["recommended_price"] - base_price
).round(2)

sensitivity_results

,scenario,recommended_price,price_change
0,Baseline,9214.99,0.00
1,Earlier departure,9921.94,706.95
2,Fewer seats,10212.02,997.03
3,Higher historical demand,9742.09,527.10
4,Higher competitor price,9687.44,472.45
5,Higher booking velocity,9633.68,418.69


## Dynamic Pricing Sensitivity Findings

The pricing engine was tested by changing one flight-state variable at a time while keeping the remaining variables fixed.

The baseline scenario produced a recommended price of ₹9,214.99.

| Scenario | Recommended Price | Change vs Baseline |
|---|---:|---:|
| Baseline | ₹9,214.99 | ₹0.00 |
| Earlier departure | ₹9,921.94 | +₹706.95 |
| Fewer seats | ₹10,212.02 | +₹997.03 |
| Higher historical demand | ₹9,742.09 | +₹527.10 |
| Higher competitor price | ₹9,687.44 | +₹472.45 |
| Higher booking velocity | ₹9,633.68 | +₹418.69 |

The model increases its predicted price when the tested demand, scarcity, timing, competitive-price, or booking-velocity signals increase.

These results demonstrate that the pricing engine responds dynamically to changes in flight-state variables.

However, these changes represent model associations rather than causal price elasticity. The dataset does not contain observed demand responses to alternative ticket prices, so the results should not be interpreted as evidence that a specific price increase will produce a specific change in bookings or revenue.